In [ ]:
# Preparing environment
import os
import sys
import random
import string
import json
import math
import numpy as np

import  nltk
nltk.download('stopwords')

from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from collections import defaultdict
from array import array

from dotenv import load_dotenv

load_dotenv()  # take environment variables from .env

[nltk_data] Downloading package stopwords to /Users/laura/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


ModuleNotFoundError: No module named 'project_progress'

In [ ]:
def build_terms(text):
    """
    Preprocesses the text fields of the document in the corpus (only`title` and `description`) by removing stop words, tokenizing, removing punctuation marks, stemming and [#TODO].

    :param text: (string) text to be processed
    :return text: List of tokens corresponding to the input text after the preprocessing
    """

    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))

    text = text.lower()  # Transform to lowercase
    text = text.split(" ")  # Tokenize the text, separating by spaces
    text = [
        word.strip(string.punctuation)
        for word in text
        if word.strip(string.punctuation).isalnum() and word not in stop_words
    ]  # eliminate the stop words [`strip` is to separate exclamation signs from words, e.g. "Hello!"" -> "Hello" + "!"]
    text = [stemmer.stem(word) for word in text]  # perform stemming

    # TODO: add more preprocessing if necessary

    return text

def join_build_terms(strings):
    """
    Builds the terms by concatenating the strings in the given list.

    :param strings: List of string. Texts to be concatenated and processed.
    :return terms: List of string, where each item is a word.
    """
    arg = " ".join(strings)
    return build_terms(arg)

In [ ]:
def create_index_tf_idf(corpus):
    """
    Implement the inverted index and compute tf, df and idf

    Argument:
    corpus -- 

    #TODO: adapt to our version

    Returns:
    index - the inverted index (implemented through a Python dictionary) containing terms as keys and the corresponding
    list of document these keys appears in (and the positions) as values.
    index2title - a mapping of article pid to its title
    tf - normalized term frequency for each term in each document
    df - number of documents each term appear in
    idf - inverse document frequency of each term
    """

    index = defaultdict(dict)
    tf = defaultdict(dict)
    df = defaultdict(dict)
    idf = defaultdict(dict)
    index2title = {}
    num_articles = len(corpus)

    for doc in list(corpus.values()):
        index2title[doc.pid] = doc.title
        # For each field to be considered in the index, get its terms
        title_description = join_build_terms([doc.title, doc.description])  # Pre-process the `title` and `description`
        brand_terms = join_build_terms([doc.brand])
        category_terms = join_build_terms([doc.category])
        sub_category_terms = join_build_terms([doc.sub_category])
        seller_product_details = join_build_terms([doc.seller, " ".join([detail for detail in doc.product_details.values()])])  # Pre-process the `title` and `description`

        # Fields and target terms to process
        fields = ['title_description', 'brand', 'category', 'sub_category', 'seller_product_details']
        target = [title_description, brand_terms, category_terms, sub_category_terms, seller_product_details]

        # Initialize a temporal dictionary to store the index terms for the current article
        current_article_index = defaultdict(dict)

        # Create the index for the current article
        # For each field we consider in the index
        for idx, field in enumerate(fields):    
            # For each term in the target field
            for position, term in enumerate(target[idx]):   
                try:
                    # Add the new found term's position to the dict
                    current_article_index[term][field][1].append(position)  
                except:
                    # Create the entry for the term and field with the term's position if it didn't exist
                    current_article_index[term][field] = [doc.pid, [position]]  

        for field in fields:
            norm = 0
            for term, positions in current_article_index.items():
                norm += len(positions[field][1])**2
            norm = math.sqrt(norm)

            for term, positions in current_article_index.items():
                # Compute term frequency and document frequency of each term per category
                try:
                    tf[term][field].append(np.round(len(positions[field][1])/norm, 4))
                    df[term][field] += 1
                # If it's a term we haven't seen, create a new term frequency and document frequency entry
                except:
                    tf[term][field] = [np.round(len(positions[field][1])/norm, 4)]
                    df[term][field] = 1

                # In the practice, the tf/df and the index were in separate loops. Both codes are now in one 
                # loop to avoid reading the same twice
                # Join the current article's index with the global index
                try:
                    index[term][field].append(positions[field])  # Add the array of positions ("[id, [[0],[1]]]"") in the given term and field
                except:
                    index[term][field] = [positions[field]]      # Create the entry for the term and the field with the array of positions


    for term, posting_fields in df:
        for field, posting in posting_fields:
            idf[term][field] = np.round(np.log(float(num_articles / df[term])), 4)

    return index, index2title, tf, df, idf

In [ ]:
    json_path = os.getenv("DATA_FILE_PATH")
    corpus = corpus_df_loading(json_path)

In [ ]:
print("")